# HopeGait GEC Training Notebook

This notebook follows the gated HopeGait methodology:

1. Create true GEC pairs by running Gipformer over ViMedCSS audio.
2. Measure raw ASR before training.
3. Run the retrieval/LLM correction baseline.
4. Train QLoRA GEC only on audited `raw_asr -> gold_text` pairs.
5. Accept the trained GEC only if validation and hard split metrics improve.

In [ ]:
# Use a GPU runtime: Runtime -> Change runtime type -> T4/L4/A100 GPU
!nvidia-smi

In [ ]:
!pip install -q "datasets[audio]" soundfile huggingface_hub sherpa-onnx numpy jiwer evaluate
!pip install -q transformers accelerate peft trl bitsandbytes sentencepiece

In [ ]:
# Clone or upload the HopeGait repo, then set REPO_DIR.
# In Colab, this can be a GitHub clone, a Drive-mounted folder, or a zip upload.
from pathlib import Path
REPO_DIR = Path('/content/HopeGait')
assert REPO_DIR.exists(), f'Update REPO_DIR. Not found: {REPO_DIR}'
%cd {REPO_DIR}

In [ ]:
# Create a small smoke-test pair file first. Remove --limit-per-split for full runs.
!PYTHONPATH=apps/api python scripts/create_gec_pairs.py \
  --output artifacts/gec_pairs/vimedcss_gipformer_pairs_smoke.jsonl \
  --limit-per-split 20

In [ ]:
# Raw Gipformer baseline. This must exist before training.
!PYTHONPATH=apps/api python scripts/evaluate_corrections.py \
  --input artifacts/gec_pairs/vimedcss_gipformer_pairs_smoke.jsonl \
  --prediction-column raw_asr

In [ ]:
# Optional LLM/RAG baseline. Set LLM_PROVIDER=openai_compatible and credentials for a real LLM.
%env LLM_PROVIDER=offline
!PYTHONPATH=apps/api python scripts/run_llm_rag_baseline.py \
  --input artifacts/gec_pairs/vimedcss_gipformer_pairs_smoke.jsonl \
  --output artifacts/baselines/llm_rag_smoke.jsonl

In [ ]:
!PYTHONPATH=apps/api python scripts/evaluate_corrections.py \
  --input artifacts/baselines/llm_rag_smoke.jsonl \
  --prediction-column corrected_text

In [ ]:
# Full pair creation. Run this after the smoke test passes.
# !PYTHONPATH=apps/api python scripts/create_gec_pairs.py \
#   --output artifacts/gec_pairs/vimedcss_gipformer_pairs.jsonl

In [ ]:
# QLoRA GEC training. Default: Qwen3-4B; fallback: Qwen2.5-3B on OOM.
!PYTHONPATH=apps/api python scripts/train_gec_lora.py \
  --pairs artifacts/gec_pairs/vimedcss_gipformer_pairs_smoke.jsonl \
  --output-dir artifacts/gec_lora/qwen_gec_smoke \
  --max-steps 20

## Acceptance Gate

Do not wire the trained adapter into the HopeGait app unless it beats both raw ASR and the LLM/RAG baseline on validation and hard split code-switched term recall/F1, while not degrading WER/CER or number/unit preservation.